# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`This notebook provides an example for loading and exploring a dataset using the `mlcroissant` library.### Dataset SourceThe dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed!pip install mlcroissant

## 1. Data LoadingLoad metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pd# Define the dataset Croissant schema URLurl = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'# Load the dataset objectdataset = mlc.Dataset(url)# Access the metadata (as an object, not as a dict)md = dataset.metadataprint(f"Name: {md.name}\nDescription: {md.description}")

## 2. Data OverviewReview available record sets, their fields, and columns. All entities are referenced by their `@id` for precise access.Croissant datasets structure tabular/structured data into one or more **Record Sets**. We'll enumerate them below.

In [ ]:
# List all available Record Sets and their fields/columns by @idrecordset_ids = []for rs in md.record_sets:    print(f"Record Set: '{rs.name}' | @id: '{rs.id}'")    recordset_ids.append(rs.id)    print('  Fields:')    for field in rs.fields:        print(f"    - {field.name if hasattr(field, 'name') else '[Unnamed]'} (@id: {field.id})")    if rs.columns:        print('  Columns:')        for col in rs.columns:            colname = col.name if hasattr(col, 'name') else '[Unnamed]'            print(f"    - {colname} (@id: {col.id})")    print('-'*40)# Optionally, print all available record set IDs for referenceprint("Available Record Set @ids:", recordset_ids)

## 3. Data ExtractionExtract records for each Record Set by supplying the `@id`. All operations reference `@id` fields.

In [ ]:
# Load each record set into a pandas DataFrame, using the record set `@id`dataframes = {}for rs_id in recordset_ids:    print(f"Loading records for Record Set @id: {rs_id}")    records = list(dataset.records(record_set=rs_id))    if records:        df = pd.DataFrame(records)        dataframes[rs_id] = df        print(f"Shape: {df.shape}")        print(f"Columns: {df.columns.tolist()}")        print(df.head())    else:        print("[WARNING] No records found.")    print('-'*60)# For the rest of the notebook, pick the first record set for demonstrationselected_recordset_id = recordset_ids[0] if recordset_ids else Noneif selected_recordset_id is not None:    print(f"Proceeding with first record set: {selected_recordset_id}")    print("Columns available:", dataframes[selected_recordset_id].columns.tolist())else:    print("[ERROR] No record sets found!")

## 4. Exploratory Data Analysis (EDA)Apply common data processing steps. As an example, filter records by a numeric column, normalize values, and group data if a categorical field is present._**Note**: All columns/fields used are referenced by their `@id`._

In [ ]:
import numpy as np# REVIEW: Pick a numeric field by its @id (choose from previous cell output).# (For illustration, let's try to find a suitable numeric field among available columns)if selected_recordset_id is not None:    df = dataframes[selected_recordset_id]    # Guess possible numeric columns by dtype    possible_numeric = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]    if not possible_numeric:        # Fallback: try parsing columns as numeric if not already        for col in df.columns:            try:                df[col] = pd.to_numeric(df[col])            except:                pass        possible_numeric = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]    print("Possible numeric fields identified:", possible_numeric)    # Use the first numeric field, or fallback to user input    if possible_numeric:        numeric_field_id = possible_numeric[0]        print(f"Using field '{numeric_field_id}' for numeric filtering/normalization.")        # Example: filter values greater than the 75th percentile        threshold = df[numeric_field_id].quantile(0.75)        filtered_df = df[df[numeric_field_id] > threshold].copy()        print(f"Filtered records with {numeric_field_id} > {threshold}")        print(filtered_df.head())        # Normalize        norm_col = f"{numeric_field_id}_normalized"        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()        print(f"Normalized {numeric_field_id} for filtered records:")        print(filtered_df[[numeric_field_id, norm_col]].head())        # Try grouping by a categorical field (noted by dtype == object and more than 1 unique value)        possible_group_fields = [col for col in df.columns if df[col].nunique() > 1 and df[col].dtype == object]        if possible_group_fields:            group_field_id = possible_group_fields[0]            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)            print(f"Grouped filtered records by '{group_field_id}':")            print(grouped_df.head())        else:            print("No suitable categorical field found for grouping.")    else:        print("No numeric field found; further analysis cannot be performed in this cell.")else:    print("No selected record set to perform EDA.")

## 5. VisualizationVisualize data distributions or relationships between fields in the dataset.We'll plot the distribution of the chosen numeric field, and if grouping was performed, the mean value per group.

In [ ]:
import matplotlib.pyplot as pltif selected_recordset_id is not None and 'numeric_field_id' in locals():    # Distribution of the numeric field    df = dataframes[selected_recordset_id]    plt.figure(figsize=(7, 4))    df[numeric_field_id].hist(bins=30, color='skyblue')    plt.title(f"Distribution of field: {numeric_field_id}")    plt.xlabel(numeric_field_id)    plt.ylabel('Count')    plt.show()    # If grouping was performed, plot group means    if 'grouped_df' in locals():        grouped_df.plot(kind='bar', legend=False, figsize=(10,4))        plt.ylabel(f"Mean {numeric_field_id}")        plt.title(f"Mean {numeric_field_id} by group: {group_field_id}")        plt.xticks(rotation=45)        plt.show()else:    print("No numeric field selected for visualization.")

## 6. Conclusion- We loaded the FAIR<sup>2</sup> "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset via its Croissant schema using `mlcroissant`.- We explored its record set structure via `@id` references, loaded records, and inspected data fields.- We performed basic exploratory analysis: filtering, normalization, and grouping of numeric data fields.- We visualized distributions and analyzed grouped means as an introduction for deeper analyses._Further steps may include detailed biological analysis, advanced statistical modeling, or downstream ML tasks with this data._